In [1]:
import re
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.7.1+cu128
Device: cuda
GPU: NVIDIA GeForce GTX 1650


In [2]:
embedding_dim = 50
glove_path = "glove.6B.50d.txt"


pretrained = {}

with open(glove_path, "r", encoding="utf-8") as file:
    for line in file:
        values = line.rstrip().split()
        word = values[0]
        vector = np.asarray(values[1:], dtype=np.float32)

        if vector.shape == (embedding_dim,):
            pretrained[word] = vector

print("Loaded vectors:", len(pretrained))
print("Example vector shape:", pretrained["good"].shape)

Loaded vectors: 400000
Example vector shape: (50,)


In [3]:
# preparing small dataset 

texts = [
    "this movie is good",
    "this film is excellent",
    "the story is wonderful",
    "i really enjoyed this movie",
    "this movie is bad",
    "the film is terrible",
    "the story is boring",
    "i hated this movie",
]

labels = [1, 1, 1, 1, 0, 0, 0, 0]

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_texts = [tokenize(text) for text in texts]

print(tokenized_texts[0])
print("Number of samples:", len(texts))

['this', 'movie', 'is', 'good']
Number of samples: 8


In [4]:
#  Build the vocabulary and check coverage
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for tokens in tokenized_texts:
    for word in tokens:
        if word not in vocab:
            vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))
print("Example IDs:", {
    word: vocab[word]
    for word in ["good", "movie", "bad"]
})

Vocabulary size: 18
Example IDs: {'good': 5, 'movie': 3, 'bad': 14}


In [5]:
# check which words exist in the pretrained file.

matched_words = [
    word for word in vocab
    if word not in (PAD_TOKEN, UNK_TOKEN)
    and word in pretrained
]

missing_words = [
    word for word in vocab
    if word not in (PAD_TOKEN, UNK_TOKEN)
    and word not in pretrained
]

real_vocab_size = len(vocab) - 2
coverage = len(matched_words) / real_vocab_size * 100

print(f"Matched words: {len(matched_words)}")
print(f"Missing words: {len(missing_words)}")
print(f"Vocabulary coverage: {coverage:.2f}%")
print("Missing examples:", missing_words[:10])

Matched words: 16
Missing words: 0
Vocabulary coverage: 100.00%
Missing examples: []


In [6]:
# Construct the embedding matrix

embedding_matrix = np.random.normal(
    loc=0.0,
    scale=0.1,
    size=(len(vocab), embedding_dim)
).astype(np.float32)

# Padding must have a zero vector.
embedding_matrix[vocab[PAD_TOKEN]] = 0.0

# Replace known words with pretrained GloVe vectors.
for word, idx in vocab.items():
    if word in (PAD_TOKEN, UNK_TOKEN):
        continue

    if word in pretrained:
        embedding_matrix[idx] = pretrained[word]

print("Embedding matrix shape:", embedding_matrix.shape)
print("Padding vector:", embedding_matrix[vocab[PAD_TOKEN]])
print("Good vector matches GloVe:",
      np.allclose(embedding_matrix[vocab["good"]], pretrained["good"])
      if "good" in pretrained else "Word not found")

Embedding matrix shape: (18, 50)
Padding vector: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0.]
Good vector matches GloVe: True


In [7]:
# Convert sentences into padded sequences

max_len = max(len(tokens) for tokens in tokenized_texts)

def encode(tokens):
    ids = [
        vocab.get(word, vocab[UNK_TOKEN])
        for word in tokens
    ]

    ids = ids[:max_len]
    ids += [vocab[PAD_TOKEN]] * (max_len - len(ids))

    return ids

X = torch.tensor(
    [encode(tokens) for tokens in tokenized_texts],
    dtype=torch.long
)

y = torch.tensor(labels, dtype=torch.float32)

print("Input shape:", X.shape)
print("Labels shape:", y.shape)
print("First encoded sample:", X[0])

Input shape: torch.Size([8, 5])
Labels shape: torch.Size([8])
First encoded sample: tensor([2, 3, 4, 5, 0])


In [10]:
# Create a PyTorch embedding layer

weights = torch.from_numpy(embedding_matrix)

freeze_embeddings = True

embedding_layer = nn.Embedding.from_pretrained(
    weights,
    freeze=freeze_embeddings,
    padding_idx=vocab[PAD_TOKEN]
).to(device)

sample_ids = X[0].to(device)

with torch.no_grad():
    sample_vectors = embedding_layer(sample_ids)

print("Input IDs shape:", sample_ids.shape)
print("Embedding output shape:", sample_vectors.shape)
print("Device:", sample_vectors.device)

Input IDs shape: torch.Size([5])
Embedding output shape: torch.Size([5, 50])
Device: cuda:0


In [11]:
# Build a small sentiment classifier

class SentimentClassifier(nn.Module):
    def __init__(self, embedding_matrix, padding_idx=0, freeze=True):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=freeze,
            padding_idx=padding_idx
        )

        self.classifier = nn.Linear(
            embedding_matrix.shape[1], 1
        )

    def forward(self, input_ids):
        # (batch_size, sequence_length, embedding_dim)
        vectors = self.embedding(input_ids)

        # Exclude padding positions from the average.
        mask = (input_ids != self.embedding.padding_idx).unsqueeze(-1)
        mask = mask.to(vectors.dtype)

        summed = (vectors * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        pooled = summed / counts

        return self.classifier(pooled).squeeze(-1)

In [12]:
# Train and test the model

model = SentimentClassifier(
    embedding_matrix=embedding_matrix,
    padding_idx=vocab[PAD_TOKEN],
    freeze=True  # Change to False to fine-tune embeddings.
).to(device)

X_device = X.to(device)
y_device = y.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad),
    lr=0.01
)

epochs = 30

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()
    logits = model(X_device)
    loss = criterion(logits, y_device)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:02d} | Loss: {loss.item():.4f}")

Epoch 05 | Loss: 0.6543
Epoch 10 | Loss: 0.6041
Epoch 15 | Loss: 0.5576
Epoch 20 | Loss: 0.5142
Epoch 25 | Loss: 0.4746
Epoch 30 | Loss: 0.4385


In [13]:
# Now test new sentences:

test_texts = [
    "this movie is excellent",
    "this film is boring",
    "i enjoyed the story"
]

test_ids = torch.tensor(
    [encode(tokenize(text)) for text in test_texts],
    dtype=torch.long,
    device=device
)

model.eval()

with torch.no_grad():
    probabilities = torch.sigmoid(model(test_ids))

for text, probability in zip(test_texts, probabilities.cpu().tolist()):
    prediction = "Positive" if probability >= 0.5 else "Negative"
    print(f"{text!r} -> {prediction} ({probability:.3f})")

'this movie is excellent' -> Positive (0.716)
'this film is boring' -> Negative (0.438)
'i enjoyed the story' -> Positive (0.572)
